# Prepare reusable EXIT retrieval artifacts
Attach the repo as an input or clone it yourself. Run steps individually; save checkpoints before the session ends. Read retrieval/README.md for storage limits. No publishing is automatic.

In [ ]:
from pathlib import Path
import os
REPO = Path('/kaggle/working/EXIT')  # adjust to your checkout
os.chdir(REPO)

In [ ]:
%pip install -r requirements-retrieval.txt
!python -m retrieval.prepare preflight --work /tmp/retrieval-state

## Optional: restore a saved checkpoint
Uncomment and adjust the package path. Restore to writable scratch, never into Input.

In [ ]:
# !python -m retrieval.package restore --source /kaggle/input/checkpoint/package.json --parts /kaggle/input --output /tmp/retrieval-state

In [ ]:
!python -m retrieval.prepare download --source corpus --work /tmp/retrieval-downloads
!python -m retrieval.prepare download --source embeddings --work /tmp/retrieval-downloads

In [ ]:
!python -m retrieval.prepare extract --archive /tmp/retrieval-downloads/psgs_w100.tsv.gz --output /tmp/retrieval-sources
!python -m retrieval.prepare extract --archive /tmp/retrieval-downloads/wikipedia_embeddings.tar --output /tmp/retrieval-sources

In [ ]:
!python -m retrieval.prepare prepare-corpus --tsv /tmp/retrieval-sources/psgs_w100.tsv --corpus /tmp/retrieval-state/corpus.sqlite


In [ ]:
import glob
print(glob.glob('/tmp/retrieval-sources/**/*', recursive=True)[:30])
SHARDS = '/tmp/retrieval-sources/wikipedia_embeddings/passages_*'  # verify actual names
assert glob.glob(SHARDS)

In [ ]:
!python -m retrieval.prepare sample-pq --shards "{SHARDS}" --work /tmp/retrieval-state/work

In [ ]:
!python -m retrieval.prepare train-pq --work /tmp/retrieval-state/work

In [ ]:
!python -m retrieval.prepare build-index --shards "{SHARDS}" --corpus /tmp/retrieval-state/corpus.sqlite --work /tmp/retrieval-state/work --output /tmp/retrieval-artifact

In [ ]:
!python -m retrieval.prepare verify --output /tmp/retrieval-artifact

## Save before stopping
Choose either checkpoint state (to resume) or final artifact (for inference). Verify total output allowance before packaging. For oversized output, use --select-part with an immutable preserved source; see guide.

In [ ]:
# !python -m retrieval.package pack --source /tmp/retrieval-state --output /kaggle/working/checkpoint-package
# !python -m retrieval.package pack --source /tmp/retrieval-artifact --output /kaggle/working/artifact-package